# Verify the Sionna RT side of the channel emulator

The diagram (`web/spark/brief-uav.html`) says: outside the slot loop, Sionna RT ray-traces a UAV flying through an urban
street canyon between two gNBs, and sends the path gains and delays as live taps; the emulator swaps them in at a slot
boundary and samples them into its filter taps. This notebook checks exactly that part, and assumes the rest of the
setup works.

Everything it runs comes from `spark/rt/uav_rt.py`, the module the Spark imports too. Nothing is copied into the
notebook, so what passes here is what runs there.

**Run it:** in Colab, *Runtime → Change runtime type → GPU*, then *Run all*. It also runs on any CPU, more slowly.
The setup cell fetches `uav_rt.py` from the repo, or asks you to upload it.

**What you get:** a PASS or FAIL line per check, a summary table at the end, and three files:
`uav_taps.npz` (the traced route, which the publisher replays), `uav_rt_report.json` (every check with its numbers)
and `uav_rt_verify.png`.

In [ ]:
# Setup: Sionna RT 2.2.0, pyzmq, and the module. Colab already has torch, numpy and matplotlib.
import os, subprocess, sys, urllib.request

try:
    import sionna.rt, zmq, torch
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sionna-rt==2.2.0", "pyzmq", "torch"])

MODULE_URL = "https://raw.githubusercontent.com/FAAMT/ddr-bram-tx-rx/main/spark/rt/uav_rt.py"
if not os.path.exists("uav_rt.py"):
    try:
        urllib.request.urlretrieve(MODULE_URL, "uav_rt.py")
    except Exception as e:
        print(f"Could not fetch {MODULE_URL} ({e}). Upload spark/rt/uav_rt.py from the repo:")
        from google.colab import files
        files.upload()
print("module:", os.path.abspath("uav_rt.py"))

In [ ]:
import json, threading, time
import numpy as np, torch, matplotlib.pyplot as plt
import mitsuba as mi, sionna.rt as rt
from sionna.rt import PathSolver, Receiver
import uav_rt as U

ENV = {"sionna_rt": rt.__version__, "variant": mi.variant(), "torch": torch.__version__,
       "cuda": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
print(ENV)
if not mi.variant().startswith("cuda"):
    print("Sionna RT is on the CPU (LLVM). Every check still holds; only the timings differ from a GPU.")

CHECKS = []
def check(name, ok, detail, **numbers):
    CHECKS.append({"check": name, "pass": bool(ok), "detail": detail, **numbers})
    print(("PASS  " if ok else "FAIL  ") + name + "  |  " + detail)

## 1. The scene matches the slide

Two gNBs on rooftops, a UAV route that stays in the street, below the roofs on both sides. The checks cast rays
against the same geometry the solver traces.

In [ ]:
SC = U.Scenario()
scene = U.build_scene(SC)
G = np.array(SC.gnbs); W = np.array(SC.waypoints)
t_route, si_route, pos_route = U.trajectory(SC)

def cast(o, d):
    # Distance along d to the first surface, inf if none. o, d: [N, 3] or [3].
    o, d = np.broadcast_arrays(np.atleast_2d(np.asarray(o, float)), np.atleast_2d(np.asarray(d, float)))
    ray = mi.Ray3f(mi.Point3f(*[mi.Float(np.ascontiguousarray(o[:, i])) for i in range(3)]),
                   mi.Vector3f(*[mi.Float(np.ascontiguousarray(d[:, i])) for i in range(3)]))
    si = scene.mi_scene.ray_intersect(ray)
    return np.where(np.array(si.is_valid()), np.array(si.t), np.inf)

down, up = [0, 0, -1], [0, 0, 1]
below = cast(G, down); above = cast(G, up)
check("gNBs sit on rooftops", ((below > 1) & (below < 5) & np.isinf(above)).all(),
      "roof " + ", ".join(f"{b:.2f} m below gNB {i+1}" for i, b in enumerate(below)) + "; open sky above",
      roof_below_m=below.tolist())

# The route is clear of buildings with a margin, along every segment and at every trace position.
MARGIN = 1.5
clear = True
for p0, p1 in zip(W[:-1], W[1:]):
    d = p1 - p0; L = np.linalg.norm(d); d /= L
    side = np.cross(d, up); side /= np.linalg.norm(side)
    for off in (0 * side, MARGIN * side, -MARGIN * side, [0, 0, MARGIN], [0, 0, -MARGIN]):
        clear &= bool((cast(p0 + off, d) >= L).all())
ground = pos_route[:, 2] - cast(pos_route, down)
check(f"UAV route is clear of buildings by {MARGIN} m", clear and np.abs(ground).max() < 0.5,
      f"{len(W)-1} segments, {len(pos_route)} positions, all over open street")

# In the canyon: walls within 30 m on both sides, and both taller than the UAV.
tangent = np.gradient(pos_route, axis=0); tangent[:, 2] = 0
tangent /= np.linalg.norm(tangent, axis=1, keepdims=True)
side = np.cross(tangent, up)
in_canyon = np.ones(len(pos_route), bool); widths = np.zeros(len(pos_route))
for sgn in (1, -1):
    dist = cast(pos_route, sgn * side)
    wall = pos_route + sgn * side * (np.where(np.isfinite(dist), dist, 1e3)[:, None] + 0.5)
    roof = 150 - cast(wall * [1, 1, 0] + [0, 0, 150], down)
    in_canyon &= (dist < 30) & (roof > pos_route[:, 2])
    widths += np.where(np.isfinite(dist), dist, 0)
frac = in_canyon.mean()
check("UAV flies through a street canyon", frac >= 0.75,
      f"walls taller than the UAV on both sides at {100*frac:.0f}% of positions (the rest is the square it starts in); "
      f"median street width {np.median(widths[in_canyon]):.1f} m", canyon_fraction=float(frac))

In [ ]:
# Top view: roof heights, the route, the gNBs.
xs = np.arange(W[:, 0].min() - 40, G[:, 0].max() + 40, 1.0); ys = np.arange(W[:, 1].min() - 25, G[:, 1].max() + 20, 1.0)
X, Y = np.meshgrid(xs, ys)
H = 150 - cast(np.stack([X.ravel(), Y.ravel(), np.full(X.size, 150.0)], 1), down).reshape(X.shape)
fig, ax = plt.subplots(figsize=(6, 7))
im = ax.pcolormesh(X, Y, H, cmap="Greys", shading="auto"); fig.colorbar(im, ax=ax, label="roof height (m)")
ax.plot(pos_route[:, 0], pos_route[:, 1], color="#c47a00", lw=2.5, label="UAV route")
ax.scatter(G[:, 0], G[:, 1], marker="^", s=120, color="#5b2a86", zorder=3, label="gNB")
for i, g in enumerate(G):
    ax.annotate(f"gNB {i+1}", g[:2], textcoords="offset points", xytext=(-44 if i == 0 else 8, 8))
ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.legend(loc="lower right")
ax.set_title(f"munich scene: {t_route[-1]:.1f} s route at {SC.speed_mps:g} m/s, UAV at {W[:,2].min():g}-{W[:,2].max():g} m")
plt.show()

## 2. Trace the route

One trace every 50 ms of stream time (0.5 m at 10 m/s), one receiver per `PathSolver` call, as the live publisher
would trace it. The table is saved: the publisher replays it, so the emulator gets the same taps on every run.

In [ ]:
table = U.trace_route(scene, SC)
table.save("uav_taps.npz")
K, NL, P = table.a.shape
npaths = table.valid.sum(-1); gain = table.gain_db()
print(f"{K} traces, {NL} links, up to {P} paths per link, {table.meta['trace_seconds']:.1f} s of tracing")
check("every position traced, both links", K == len(t_route) and NL == 2 and np.allclose(table.pos, pos_route),
      f"{K} positions x 2 links")
check("both links have paths at every position", (npaths > 0).all(),
      f"paths per link: min {npaths.min(0).tolist()}, median {np.median(npaths, 0).tolist()}",
      outage_positions=(npaths == 0).sum(0).tolist())

## 3. The paths obey the physics

With isotropic antennas, a line-of-sight path must equal free space exactly, no path can arrive before light covers
the straight line, and no path can be stronger than free space over its own length (reflections and transmission only
lose energy). These pin down units, carrier, delays and antenna gains.

In [ ]:
lam = U.C / SC.carrier_hz
d = np.linalg.norm(table.pos[:, None, :] - G[None, :, :], axis=-1)               # [K, 2]
m = table.valid
li = np.nonzero(table.los)
gain_err = 20*np.log10(np.abs(table.a[li])) - 20*np.log10(lam / (4*np.pi*d[li[0], li[1]]))
delay_err = table.tau[li] - d[li[0], li[1]] / U.C
check("line-of-sight paths equal free space", len(li[0]) > 0 and np.abs(gain_err).max() < 0.01
      and np.abs(delay_err).max() < 1e-11,
      f"{len(li[0])} LOS paths; gain within {np.abs(gain_err).max():.1e} dB of Friis, delay within "
      f"{1e9*np.abs(delay_err).max():.1e} ns of d/c; LOS at {100*table.los.any(-1).mean(0)[0]:.0f}% / "
      f"{100*table.los.any(-1).mean(0)[1]:.0f}% of positions")

early = (table.tau - d[..., None] / U.C)[m]
check("no path arrives before the direct line", early.min() > -1e-11, f"earliest path {1e9*early.min():+.1e} ns from d/c")

excess = 20*np.log10(np.abs(table.a[m])) - 20*np.log10(lam / (4*np.pi*U.C*table.tau[m]))
check("no path beats free space over its own length", excess.max() < 0.01,
      f"{m.sum()} paths; strongest relative to free space at its length: {excess.max():+.1e} dB")

q = K // 5
first, last = gain[:q].mean(0), gain[-q:].mean(0)
check("gain rises as the UAV nears the gNBs", (last > first).all(),
      f"mean path gain, first fifth -> last fifth: link 1 {first[0]:.1f} -> {last[0]:.1f} dB, "
      f"link 2 {first[1]:.1f} -> {last[1]:.1f} dB")

## 4. Repeatability

Two traces of the same position must give the same paths. The solver's cap on candidate paths must not be dropping
any: past it, Sionna RT discards paths without a warning (while this was built, 64 receivers per call lost up to
4.6 dB out of line of sight). And the replay from the saved table must be exact, since that is what the emulator sees.

In [ ]:
from dataclasses import replace

def retrace(sc, idx):
    solver, g, n = PathSolver(), [], []
    for k in idx:
        a, _, v, _ = U.trace_paths(scene, table.pos[k:k+1], sc, solver)
        with np.errstate(divide="ignore"):
            g.append(10*np.log10((np.abs(a)**2 * v).sum(-1))[0])
        n.append(v.sum(-1)[0])
    return np.array(g), np.array(n)

idx = np.arange(0, K, 4)
g2, n2 = retrace(SC, idx)
dg = np.abs(g2 - gain[idx])
check("a second trace gives the same paths", np.array_equal(n2, npaths[idx]) and dg.max() < 0.01,
      f"{len(idx)} positions re-traced: same path count on all {2*len(idx)} links, gain within {dg.max():.1e} dB",
      retrace_db_max=float(dg.max()))

idx = np.arange(0, K, 8)
g4, n4 = retrace(replace(SC, max_paths_per_src=4 * SC.max_paths_per_src), idx)
dg = np.abs(g4 - gain[idx])
check("the solver's path cap drops nothing", np.array_equal(n4, npaths[idx]) and dg.max() < 0.01,
      f"{len(idx)} positions at 4x the cap: no extra paths, gain within {dg.max():.1e} dB")

again = U.messages(U.Table.load("uav_taps.npz"), SC)
frames = U.messages(table, SC)
check("replay from the saved table is exact", again == frames and np.array_equal(U.trajectory(SC)[1], table.sample_index),
      f"{len(frames)} messages byte-identical after save and load")

## 5. The live-taps message

The message of plan.md section 6: a JSON header, then the gains, then the delays. Checked: everything survives the
round trip, the scaling keeps the route's fading, and every malformed message is refused.

In [ ]:
A = U.scaled(table, SC)
ok_rt = True
for k, fr in enumerate(frames):
    h, links = U.decode(fr)
    ok_rt &= h["seq"] == k and h["sample_index"] == table.sample_index[k]
    for l, (a_, t_) in enumerate(links):
        tt = table.tau[k, l][m[k, l]]
        ok_rt &= np.array_equal(a_, A[k, l][m[k, l]]) and np.array_equal(t_, tt - tt.min()) and t_.min() == 0
check("every message survives encode and decode", ok_rt, f"{len(frames)} messages, gains and delays bit-exact, "
      f"each link's first path at 0 s")

E = (np.abs(A)**2).sum(-1); drop = -10*np.log10(E)
want = np.clip(gain.max(0) - gain, 0, SC.max_drop_db)
check("scaling keeps the route's fading", np.allclose(drop, want, atol=1e-3) and np.isclose(E.max(0), 1).all(),
      f"best point at unit energy per link; deepest fade {np.round(drop.max(0).astype(float), 1).tolist()} dB "
      f"(raw range {np.round((gain.max(0)-gain.min(0)).astype(float), 1).tolist()} dB, clamp {SC.max_drop_db:g} dB)")

good = frames[0]
hdr = json.loads(good[0])
bad = {
    "two frames": good[:2],
    "wrong fs_hz": [json.dumps({**hdr, "fs_hz": 30720000}).encode(), good[1], good[2]],
    "wrong version": [json.dumps({**hdr, "v": 2}).encode(), good[1], good[2]],
    "short gains": [good[0], good[1][:-8], good[2]],
    "short delays": [good[0], good[1], good[2][:-4]],
    "not JSON": [b"\xff", good[1], good[2]],
    "missing seq": [json.dumps({k: v for k, v in hdr.items() if k != "seq"}).encode(), good[1], good[2]],
    "NaN delay": [good[0], good[1], np.r_[np.frombuffer(good[2], "<f4")[:-1], np.nan].astype("<f4").tobytes()],
    "negative delay": [good[0], good[1], np.r_[np.frombuffer(good[2], "<f4")[:-1], -1e-9].astype("<f4").tobytes()],
}
refused = []
for name, fr in bad.items():
    try:
        U.decode(fr)
    except ValueError:
        refused.append(name)
check("every malformed message is refused", len(refused) == len(bad), f"{len(refused)} of {len(bad)} refused: "
      + ", ".join(refused))

## 6. From paths to filter taps

Step 3 of the loop samples (a, τ) into 16 taps at 61.44 Msps: h[l] = Σ a·sinc(l − f_s·τ), l = −6 … 9.
`sample_taps` (torch, so it runs on the emulator's GPU) must match Sionna RT's own `Paths.taps`, and 16 taps must hold
the channel's energy.

In [ ]:
errs = []
for k in (0, K // 2, K - 1):
    scene.add(Receiver("probe", position=table.pos[k].tolist()))
    p = PathSolver()(scene, max_depth=SC.max_depth, samples_per_src=SC.samples_per_src, seed=SC.seed)
    ref = p.taps(bandwidth=U.FS_HZ, l_min=U.L_MIN, l_max=U.L_MAX, normalize=False, normalize_delays=True,
                 out_type="numpy")[0, 0, :, 0, 0, :]                                  # [2, 16]
    a_, t_ = p.cir(normalize_delays=True, out_type="numpy"); v_ = p.valid.numpy()
    mine = U.sample_taps(torch.tensor(np.where(v_[0], a_[0, 0, :, 0, :, 0], 0)), torch.tensor(np.where(v_[0], t_[0], 0)))
    errs.append(float((np.abs(mine.numpy() - ref).max(-1) / np.abs(ref).max(-1)).max()))
    scene.remove("probe")
check("sample_taps matches Sionna RT's Paths.taps", max(errs) < 1e-5,
      "max relative error " + ", ".join(f"{e:.1e}" for e in errs) + " at 3 positions")

tn = np.where(m, table.tau - np.where(m, table.tau, np.inf).min(-1, keepdims=True), 0).astype(np.float32)
h16 = U.sample_taps(torch.tensor(A), torch.tensor(tn))
hw = U.sample_taps(torch.tensor(A), torch.tensor(tn), U.L_MIN, U.L_MIN + 1023)            # 16.7 us: all of it
ew = (hw.abs()**2).sum(-1)
held = ((h16.abs()**2).sum(-1) / ew).numpy()
need99 = (((hw.abs()**2).cumsum(-1) / ew[..., None]) < 0.99).sum(-1).numpy() + 1
check("16 taps hold at least 90% of each link's energy", held.min() >= 0.90,
      f"held: median {np.round(np.median(held, 0).astype(float), 4).tolist()}, worst {np.round(held.min(0).astype(float), 3).tolist()}; "
      f"taps for 99%: median {np.median(need99, 0).tolist()}, 95th pct {np.percentile(need99, 95, 0).round().tolist()}, "
      f"max {need99.max(0).tolist()}", taps_for_99pct_max=need99.max(0).tolist())

## 7. Live taps over ZMQ, swapped at slot boundaries

The publisher (XPUB on the plan's `tcp://127.0.0.1:5566`) and the emulator's side (`subscriber`, `drain`,
`TapSwitch`) run here over loopback, with the slot loop reduced to counting slots. Each message must take effect at
the first slot that starts at or after its `sample_index`, once; a malformed message must be dropped with the taps
held; a second subscriber joining must not replay old taps; and after the last message the taps must hold.

In [ ]:
ENDPOINT = "tcp://127.0.0.1:5566"
stop, ready = threading.Event(), threading.Event()
pub = threading.Thread(target=U.publish, args=(frames, ENDPOINT, stop, ready), daemon=True); pub.start()
ready.wait(5)
sub = U.subscriber(ENDPOINT); sw = U.TapSwitch()
t0 = time.time()
while len(sw.pending) < len(frames) and time.time() - t0 < 30:      # the route arrives before the stream starts
    U.drain(sub, sw); time.sleep(0.01)

n_slots = int(table.sample_index[-1] // U.SLOT) + 400
held_ok, late = True, None
for s in range(n_slots):
    U.drain(sub, sw)
    if s == 150:
        before = sw.taps.clone(); sw.offer([b"{}", b"", b""])       # malformed, mid-route
    if s == n_slots // 2:
        late = U.subscriber(ENDPOINT); time.sleep(0.3)             # a second emulator joins; the route is re-sent
    taps = sw.at_slot(s * U.SLOT)
    if s == 150:
        held_ok &= torch.equal(taps, before)
t0 = time.time()
while sw.duplicates < len(frames) and time.time() - t0 < 10:      # let the re-sent route finish arriving
    U.drain(sub, sw); time.sleep(0.01)
stop.set(); pub.join(5); sub.close(); late.close()

expect = [(k, int(-(-table.sample_index[k] // U.SLOT)) * U.SLOT) for k in range(len(frames))]
last = torch.stack([U.sample_taps(torch.tensor(A[-1, l][m[-1, l]]), torch.tensor(tn[-1, l][m[-1, l]])) for l in range(2)])
check("each message swaps in once, at the right slot", sw.log == expect and sw.swaps == len(frames) and sw.superseded == 0,
      f"{sw.swaps} swaps for {len(frames)} messages, each at the first slot at or after its sample_index "
      f"(every {round(SC.dt_s * U.FS_HZ) // U.SLOT} slots)")
check("a malformed message is dropped and the taps hold", sw.rejected == 1 and held_ok, f"rejected {sw.rejected}")
check("a re-sent route does not replay old taps", sw.duplicates == len(frames) and sw.swaps == len(frames),
      f"{sw.duplicates} duplicates ignored after a second subscriber joined")
check("after the last message the taps hold", torch.allclose(sw.taps, last) and sw.seq == len(frames) - 1,
      f"still on message {sw.seq} at slot {n_slots - 1}, {n_slots - 1 - expect[-1][1] // U.SLOT} slots after it")

## 8. What the emulator will see

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for l, c in enumerate(("#5b2a86", "#1f7a7a")):
    ax[0, 0].plot(table.t, gain[:, l], color=c, label=f"link {l+1}: gNB {l+1} -> UAV")
    ax[0, 1].plot(table.t, npaths[:, l], color=c, label=f"link {l+1}")
ax[0, 0].set(xlabel="stream time (s)", ylabel="path gain (dB)", title="Path gain along the route"); ax[0, 0].legend()
ax[0, 1].set(xlabel="stream time (s)", ylabel="paths", title="Paths per trace"); ax[0, 1].legend()
lags = np.arange(U.L_MIN, U.L_MAX + 1)
for l in range(2):
    im = ax[1, l].imshow(20*np.log10(h16[:, l].abs().numpy().T + 1e-12), aspect="auto", origin="lower",
                         extent=[table.t[0], table.t[-1], lags[0] - .5, lags[-1] + .5], vmin=-60, vmax=0, cmap="magma")
    ax[1, l].set(xlabel="stream time (s)", ylabel="tap lag l", title=f"Link {l+1}: |h[l]| (dB) the emulator filters with")
fig.colorbar(im, ax=ax[1, :], label="dB")
fig.savefig("uav_rt_verify.png", dpi=120); plt.show()

In [ ]:
# A view like the slide's: the canyon, the gNBs, the route, and the paths at mid-route.
try:
    from sionna.rt import Camera
    k = K // 2
    scene.add(Receiver("uav", position=table.pos[k].tolist(), display_radius=2.0))
    for i in range(len(G)): scene.get(f"gnb{i+1}").display_radius = 2.0
    paths = PathSolver()(scene, max_depth=SC.max_depth, samples_per_src=SC.samples_per_src, seed=SC.seed)
    dots = [f"r{i}" for i in range(0, K, 12)]
    for n, i in zip(dots, range(0, K, 12)):
        scene.add(Receiver(n, position=table.pos[i].tolist(), color=(0.77, 0.48, 0.0), display_radius=0.8))
    start, end = table.pos[0], table.pos[-1]
    ahead = (end - start) / np.linalg.norm(end - start)
    cam = Camera(position=(start - 70 * ahead + [0, 0, 75]).tolist(), look_at=(start + 0.6 * (end - start)).tolist())
    scene.render(camera=cam, paths=paths, num_samples=64, resolution=(900, 650))
    for n in dots + ["uav"]: scene.remove(n)
    plt.show()
except Exception as e:                       # the picture is for looking at; no check depends on it
    print("render skipped:", e)

## Summary

In [ ]:
passed = sum(c["pass"] for c in CHECKS)
for c in CHECKS:
    print(("PASS  " if c["pass"] else "FAIL  ") + c["check"])
print(f"\n{passed} of {len(CHECKS)} checks passed" + ("" if passed == len(CHECKS) else "  <-- look at the FAIL lines above"))
report = {"environment": ENV, "scenario": table.meta["scenario"], "trace_seconds": table.meta["trace_seconds"],
          "traces": K, "route_seconds": float(table.t[-1]), "checks": CHECKS,
          "files": ["uav_taps.npz", "uav_rt_report.json", "uav_rt_verify.png"]}
with open("uav_rt_report.json", "w") as f:
    json.dump(report, f, indent=1, default=float)
print("wrote", ", ".join(report["files"]))

In [ ]:
# Colab: download the three files.
try:
    from google.colab import files
    for f in ("uav_taps.npz", "uav_rt_report.json", "uav_rt_verify.png"):
        files.download(f)
except ImportError:
    pass